## 15.0 PPM Dual-Backbone

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import random
from PIL import Image

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

import warnings
warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"➔ Execution device: {device}")

DATA_DIR = "/kaggle/input/competitions/soil-grain-size-from-photos"
TRAIN_LABELS_PATH = os.path.join(DATA_DIR, "Training_labels_without_H374.csv")
PPM_PATH = os.path.join(DATA_DIR, "ppm.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "Training-All_Photos_without_H374")
TEST_IMG_DIR = os.path.join(DATA_DIR, "Test_All_Photos")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")

def build_image_map(directory):
    img_map = {}
    if not os.path.exists(directory):
        return img_map
    for root, _, files in os.walk(directory):
        for file in files:
            name_idx = file.rfind('.')
            if name_idx != -1:
                sample_id = file[:name_idx].lower()
                img_map[sample_id] = os.path.join(root, file)
    return img_map

train_img_map = build_image_map(TRAIN_IMG_DIR)
test_img_map = build_image_map(TEST_IMG_DIR)

# Еталонні бекбони
effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
extractor_eff = nn.Sequential(effnet.features, effnet.avgpool, nn.Flatten()).to(device).eval()

resnet = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
extractor_res = nn.Sequential(*list(resnet.children())[:-1], nn.Flatten()).to(device).eval()

tensor_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def normalize_string(s):
    s = str(s).lower().strip()
    s = s.replace('ü', 'ue').replace('ö', 'oe').replace('ä', 'ae').replace('ß', 'ss')
    s = s.replace(',', '_').replace('-', '_').replace(' ', '_')
    while '__' in s: s = s.replace('__', '_')
    return s

def extract_pristine_features(df, img_map, ppm_lookup, global_mean_ppm, set_name="Dataset"):
    features_list = []
    print(f"\nRunning Pristine 15.0 PPM Extraction for: {set_name}")
    TARGET_PPM = 15.0 
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        sample_id = str(row["sample_id"])
        norm_sample_id = normalize_string(sample_id)
        
        matching_paths = [p for k, p in img_map.items() if (norm_sample_id in normalize_string(k) or normalize_string(k) in norm_sample_id or norm_sample_id.replace('hpc_', '') in normalize_string(k))]
        feat = {"sample_id": row["sample_id"]}
        
        if matching_paths:
            eff_embs, res_embs = [], []
            all_edges = []
            
            for img_path in matching_paths:
                img = cv2.imread(img_path)
                if img is not None:
                    h, w = img.shape[0], img.shape[1]
                    current_ppm = ppm_lookup.get((w, h), global_mean_ppm)
                    
                    scale_factor = TARGET_PPM / current_ppm
                    img_scaled = cv2.resize(img, (int(w * scale_factor), int(h * scale_factor)))
                    sh, sw = img_scaled.shape[0], img_scaled.shape[1]
                    
                    if sh < 224 or sw < 224:
                        img_scaled = cv2.resize(img_scaled, (224, 224))
                        sh, sw = img_scaled.shape[0], img_scaled.shape[1]
                    
                    gray = cv2.cvtColor(img_scaled, cv2.COLOR_BGR2GRAY)
                    all_edges.append(np.mean(cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 30, 100) > 0))
                    
                    # Наші найкращі 5 внутрішніх безпечних патчів з плоским усередненням
                    cy, cx = sh // 2, sw // 2
                    inner_shifts = [(cy, cx), (cy - 45, cx), (cy + 45, cx), (cy, cx - 45), (cy, cx + 45)]
                    img_rgb = cv2.cvtColor(img_scaled, cv2.COLOR_BGR2RGB)
                    
                    for y_c, x_c in inner_shifts:
                        y1 = max(0, min(sh - 224, y_c - 112))
                        x1 = max(0, min(sw - 224, x_c - 112))
                        patch = img_rgb[y1:y1+224, x1:x1+224]
                        tensor_patch = tensor_transform(Image.fromarray(patch)).unsqueeze(0).to(device)
                        
                        with torch.no_grad():
                            emb_eff = extractor_eff(tensor_patch).cpu().numpy().flatten()
                            emb_res = extractor_res(tensor_patch).cpu().numpy().flatten()
                        eff_embs.append(emb_eff)
                        res_embs.append(emb_res)
            
            if eff_embs:
                feat["edge_density"] = np.mean(all_edges)
                feat["ppm"] = current_ppm
                
                avg_eff = np.mean(eff_embs, axis=0)
                avg_res = np.mean(res_embs, axis=0)
                
                for i, val in enumerate(avg_eff): feat[f"eff_{i}"] = val
                for i, val in enumerate(avg_res): feat[f"res_{i}"] = val
            else: feat["edge_density"] = 0.0; feat["ppm"] = global_mean_ppm
        else: feat["edge_density"] = 0.0; feat["ppm"] = global_mean_ppm
            
        features_list.append(feat)
    return pd.DataFrame(features_list)

df_train_labels = pd.read_csv(TRAIN_LABELS_PATH)
df_sub = pd.read_csv(SAMPLE_SUB_PATH)
df_ppm = pd.read_csv(PPM_PATH)

ppm_lookup = df_ppm.set_index(["width", "height"])["ppm"].to_dict()
global_mean_ppm = df_ppm["ppm"].mean()

X_train_df = extract_pristine_features(df_train_labels, train_img_map, ppm_lookup, global_mean_ppm, "Train Set")
X_test_df = extract_pristine_features(df_sub, test_img_map, ppm_lookup, global_mean_ppm, "Test Set")
print("➔ Clean 1792 features matrix extraction complete!")

➔ Execution device: cpu
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 148MB/s]


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 140MB/s]



Running Pristine 15.0 PPM Extraction for: Train Set


100%|██████████| 25/25 [01:29<00:00,  3.59s/it]



Running Pristine 15.0 PPM Extraction for: Test Set


100%|██████████| 10/10 [00:41<00:00,  4.17s/it]

➔ Clean 1792 features matrix extraction complete!


## 10-Seed Coordinated Ridge + MultiTaskElasticNet

In [2]:
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, MultiTaskElasticNet
from sklearn.metrics import median_absolute_error

target_cols = [col for col in df_train_labels.columns if col != "sample_id"]

common_cols = list(set(X_train_df.columns) & set(X_test_df.columns))
feature_cols = [col for col in common_cols if col != "sample_id"]

X = X_train_df[feature_cols].fillna(0)
Y = df_train_labels[target_cols].values
X_test = X_test_df[feature_cols].fillna(0)

# Гранд-пул 10 насінь для тотального контролю дисперсії
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
ridge_alphas = np.logspace(-1, 5, 40)

accumulated_test_preds = np.zeros((len(X_test), Y.shape[1]))
global_seed_scores = []

print(f"➔ Training 10-Seed Dual-Engine Coordinated Blend...")

for seed in SEEDS:
    best_ridge_alpha = None
    best_ridge_medae = float('inf')
    
    # 1. Швидкий пошук найкращого Alpha для Ridge
    for alpha in ridge_alphas:
        oof_temp = np.zeros(Y.shape)
        kf = KFold(n_splits=5, shuffle=True, random_state=seed)
        for train_idx, val_idx in kf.split(X, Y):
            scaler = StandardScaler()
            X_tr_s = scaler.fit_transform(X.iloc[train_idx])
            X_va_s = scaler.transform(X.iloc[val_idx])
            
            model = Ridge(alpha=alpha, random_state=seed)
            model.fit(X_tr_s, Y[train_idx])
            oof_temp[val_idx] = model.predict(X_va_s)
            
        oof_temp = np.sort(np.clip(oof_temp, 0.0, 100.0), axis=1)
        score = median_absolute_error(Y, oof_temp)
        if score < best_ridge_medae:
            best_ridge_medae = score; best_ridge_alpha = alpha
            
    # 2. Крос-валідація для сумісного MultiTaskElasticNet (L1/L2 спадщина)
    oof_enet = np.zeros(Y.shape)
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    for train_idx, val_idx in kf.split(X, Y):
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X.iloc[train_idx])
        X_va_s = scaler.transform(X.iloc[val_idx])
        
        # alpha=0.5, l1_ratio=0.1 дає делікатне занулення абсолютно трешових фічей
        model_enet = MultiTaskElasticNet(alpha=0.5, l1_ratio=0.1, max_iter=2000, random_state=seed)
        model_enet.fit(X_tr_s, Y[train_idx])
        oof_enet[val_idx] = model_enet.predict(X_va_s)
        
    oof_enet = np.sort(np.clip(oof_enet, 0.0, 100.0), axis=1)
    
    # 3. Валідація збалансованого бленду поточного насіння
    seed_oof_blend = 0.5 * oof_temp + 0.5 * oof_enet
    seed_oof_blend = np.sort(np.clip(seed_oof_blend, 0.0, 100.0), axis=1)
    seed_medae = median_absolute_error(Y, seed_oof_blend)
    
    print(f"Seed {seed} -> Ridge Alpha: {best_ridge_alpha:.2f} | Coordinated Blend MedAE: {seed_medae:.4f}")
    global_seed_scores.append(seed_medae)
    
    # 4. Фінальне навчання моделей поточного Seed на всьому об'ємі даних
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)
    
    final_ridge = Ridge(alpha=best_ridge_alpha, random_state=seed)
    final_ridge.fit(X_scaled, Y)
    pred_test_ridge = final_ridge.predict(X_test_scaled)
    
    final_enet = MultiTaskElasticNet(alpha=0.5, l1_ratio=0.1, max_iter=2000, random_state=seed)
    final_enet.fit(X_scaled, Y)
    pred_test_enet = final_enet.predict(X_test_scaled)
    
    # Пуллінг прогнозів поточного насіння в загальний мішок
    seed_test_blend = 0.5 * pred_test_ridge + 0.5 * pred_test_enet
    accumulated_test_preds += seed_test_blend / len(SEEDS)

print(f"\n➔ Global 10-Seed MultiTask Ensemble MedAE: {np.mean(global_seed_scores):.4f}")

# Фінальний кумулятивний пост-процесинг (Суворі закони CDF)
final_test_preds = np.clip(accumulated_test_preds, 0.0, 100.0)
final_test_preds = np.sort(final_test_preds, axis=1)

submission = pd.DataFrame(final_test_preds, columns=target_cols)
submission.insert(0, "sample_id", df_sub["sample_id"])

submission.to_csv("submission.csv", index=False)
print("\n➔ Coordinated Ridge + MultiTaskElasticNet submission.csv compiled successfully!")
display(submission.head())

➔ Training 10-Seed Dual-Engine Coordinated Blend...
Seed 42 -> Ridge Alpha: 10.00 | Coordinated Blend MedAE: 10.2592
Seed 43 -> Ridge Alpha: 0.10 | Coordinated Blend MedAE: 9.9475
Seed 44 -> Ridge Alpha: 58.78 | Coordinated Blend MedAE: 10.1231
Seed 45 -> Ridge Alpha: 41.25 | Coordinated Blend MedAE: 9.3047
Seed 46 -> Ridge Alpha: 492.39 | Coordinated Blend MedAE: 10.7547
Seed 47 -> Ridge Alpha: 58.78 | Coordinated Blend MedAE: 10.8516
Seed 48 -> Ridge Alpha: 0.10 | Coordinated Blend MedAE: 9.7883
Seed 49 -> Ridge Alpha: 119.38 | Coordinated Blend MedAE: 10.5112
Seed 50 -> Ridge Alpha: 170.13 | Coordinated Blend MedAE: 10.2246
Seed 51 -> Ridge Alpha: 58.78 | Coordinated Blend MedAE: 10.1055

➔ Global 10-Seed MultiTask Ensemble MedAE: 10.1871

➔ Coordinated Ridge + MultiTaskElasticNet submission.csv compiled successfully!


,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS6-3,1.166832,2.073132,5.031209,30.838733,60.838663,63.736786,72.637984,85.592616,96.338026,100.000000,100.0
1,HPC_Audorfring,0.000000,0.000000,4.673444,19.480737,42.581080,48.883444,50.418504,61.191403,86.669439,98.640192,100.0
2,HPC_Muenster_BS6_9_0-10m,0.000000,0.000000,0.000000,0.000000,0.000000,2.364277,9.939289,37.729752,79.681535,97.719519,100.0
3,HPC_Airbus BS10-4bis7,1.784602,3.716475,12.100518,43.148860,75.781359,79.895782,88.147185,97.238762,99.971315,100.000000,100.0
4,HPC_Airbus BS12-5bis8,3.953067,5.652959,13.103741,46.687105,80.435022,81.443251,85.504271,92.279977,98.402204,100.000000,100.0
